# `ratio_study` - 04 Advanced diagnostics

Machine-learning inequity diagnostics (section 5) and sales-chasing detection (section 7).

| Notebook | Covers |
|---|---|
| [01 Basics](01_basics.ipynb) | Sections 1-2: indicators, confidence intervals, compliance checks, IQR trimming, charts, comparisons |
| [02 Spatial](02_spatial.ipynb) | Section 3: ratio maps, Moran's I, hot/cold spots, the VEM map |
| [03 Reporting](03_reporting.ipynb) | Section 4: PDF / XLSX / CSV / PNG output |
| [04 Advanced diagnostics](04_advanced_diagnostics.ipynb) | Sections 5 and 7: ML / SHAP inequity diagnostics and sales-chasing detection |
| [05 Vertical equity](05_vertical_equity.ipynb) | Section 6: VEI, the Vertical Equity Panel, VEM, the Gini family |

All five use the anonymised sample dataset bundled with the package - no data preparation required.
Section numbers are kept from the original single-notebook tour so cross-references still line up.

## Setup

Section 5 needs the ML extra; section 7 needs only the core install.

```
pip install "ratio_study[ml]"
```

In [ ]:
# Uncomment on first run.
# %pip install "ratio_study[ml]"

In [ ]:
import ratio_study as rs
import pandas as pd
import numpy as np
from pathlib import Path
from ratio_study import ml_diagnostics as ml       # section 5 - needs the [ml] extra

# inline plotting must be enabled AFTER importing ratio_study
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

sample_df = rs.load_sample_dataset()

MARKET_VALUE = 'time_adjusted_sale_price'   # the market value proxy
ASSESSMENT   = 'assessment_value_new'       # the value under test (current CAMA model)
PRIOR_MODEL  = 'assessment_value_old'       # a previous model, for comparisons

sample_df['asr'] = sample_df[ASSESSMENT] / sample_df[MARKET_VALUE]   # ratio column, needed by the spatial and VE functions

results_directory = Path().resolve()   # outputs are written here
print('results will be written to:', results_directory)

print('ratio_study version:', rs.__version__)
print(f'{len(sample_df):,} sales, {sample_df.shape[1]} columns')

---
# 5. Machine learning for ratio analysis

Every indicator so far tests **one** hypothesis at a time: does the ratio move with price
(VEI, PRB), does it differ across neighbourhoods (grouped COD), is it spatially clustered
(Moran's I). Real inequity is often non-linear and interactive — a defect that only bites
for small bungalows in one part of town will pass all of those tests individually.

The idea, following **Bidanset & Wehrli** (GIS/Valuation Technologies 2025) and **Wehrli**
(IAAO/IPTI MAVS 2026), inverts the question:

> Fit a flexible model $\widehat{ASR} = F(X)$ on property characteristics.
> **If the ratio were equitable, $X$ should not predict it at all.**
> Whatever the model *can* predict is inequity, and SHAP says which characteristics carry it.

This is a natural extension of the Standard's instruction to check that valuation levels are
consistent across property characteristics before testing vertical equity — done for all
characteristics at once, and without assuming the relationship is linear.

Two design points worth understanding:

- **Sale price is deliberately excluded from $X$.** Since $ASR = AV/SP$, a model given $SP$
  need only reconstruct $AV$ from $X$ — which any competent CAMA model makes easy. The
  resulting R² would measure how well $X$ mimics the assessment model, not anything about
  equity. Price-related bias is the VEI's and PRB's job.
- **The R² is cross-fitted (out-of-fold).** An in-sample R² would be near 1 for any boosted
  tree, and would mean nothing. This is also the Standard's overfitting concern
  in a different guise.

Needs the ML extra: `pip install "ratio_study[ml]"`.

In [ ]:
CHARACTERISTICS = ['living_area', 'lotsize', 'frontage', 'depth',
                   'quality_class', 'condition', 'baths', 'bedrooms',
                   'structure_type', 'property_type',
                   'neighbourhood', 'sale_month']

diag = ml.asr_predictability(sample_df,
                             market_value_header=MARKET_VALUE,
                             assessment_header=ASSESSMENT,
                             independent_variables=CHARACTERISTICS,
                             group_by='neighbourhood',
                             shap_sample_pct=0.20,
                             random_state=7)

print(f"Predictable share of the ASR (out-of-fold R2): {diag['predictability']:.1%}")
print(diag['attribution'])

## 5.1 How much? — the headline

`summary` states the result and how to read it. Low predictability is the good outcome: it
means the ratio looks like noise with respect to property characteristics, which is what
equity looks like.

In [ ]:
diag['summary']

## 5.2 By what? — SHAP importance

Mean absolute SHAP per characteristic, ranked. Read a value as *"this characteristic moves
the ratio by about this much"*, in ratio points.

In [ ]:
diag['shap_importance']

In [ ]:
ml.shap_importance_plot(diag, top_n=12)

### The beeswarm: direction as well as magnitude

One dot per property per characteristic. Horizontal position is the SHAP value — how far
that characteristic pushed that property's ratio. Colour is the characteristic's own value.

A clean left-to-right colour gradient is the signature of systematic bias: *high values of
this characteristic consistently push the ratio one way*.

In [ ]:
ml.shap_beeswarm_plot(diag, top_n=12)

### Dependence: the shape of one characteristic's effect

Plotting a characteristic's SHAP value against its own value exposes the non-linearity that
a PRB regression cannot see — thresholds, plateaus, and reversals. The Standard lists exactly
this as a PRB weakness: *"the method assumes a linear relationship … and its coefficient can
be misinterpreted should that not be the case."*

In [ ]:
top_feature = diag['shap_importance'].iloc[0]['feature']
print('most influential characteristic:', top_feature)

ml.shap_dependence_plot(diag, feature=top_feature)

In [ ]:
# Any characteristic can be interrogated the same way.
ml.shap_dependence_plot(diag, feature='living_area')

### Waterfall: one property, fully decomposed

For an appeal or a case review: this property's predicted ratio, broken into the
contribution of each characteristic, starting from the base rate.

> Remember — ratio study statistics do not judge an individual property. This chart
> explains the *model's* behaviour at one point, not that property's correct value.

In [ ]:
ml.shap_waterfall_plot(diag, row=0, top_n=12)

## 5.3 Subgroup diagnostics: CMBI and CDR

Two per-group statistics that pair with the grouped ratio tables from section 2.6:

- **CMBI** (Conditional Mean Bias Index) — how far a group's mean ratio sits from the
  overall mean. A *level* problem in that group — the Standard's five-percent test.
- **CDR** (Conditional Dispersion Ratio) — the group's ratio variance over the overall
  variance. Above 1 means the model is *less consistent* inside that group. A *uniformity*
  problem; the COD story.

Groups under `min_group_n` (30) are flagged unreliable rather than dropped, so a small
stratum never quietly disappears from the diagnosis.

In [ ]:
cmbi_cdr = diag['cmbi_cdr']

print(f'{len(cmbi_cdr)} neighbourhoods assessed\n')
cmbi_cdr.head(15)

In [ ]:
# Neighbourhoods that are both biased and inconsistent - the ones to look at first.
worst = cmbi_cdr[cmbi_cdr['reliable']].nlargest(10, 'cmbi')
(worst.style.format({'group_mean': '{:.4f}', 'cmbi': '{:.4f}',
                     'variance': '{:.5f}', 'cdr': '{:.3f}'})
      .background_gradient(subset=['cmbi', 'cdr'], cmap='Oranges')
      .set_caption('Largest conditional mean bias, reliable groups only'))

## 5.4 Distance correlation

**Distance correlation** is zero *if and only if* two variables are independent — unlike
Pearson correlation, which is blind to non-linear dependence. Run per characteristic
against the ratio, it is a model-free screen: anything with a non-trivial `dcorr` is
related to the ratio somehow, whatever the shape of the relationship.

Useful as a cross-check on SHAP, since it makes no modelling assumptions at all.

In [ ]:
diag['dcorr']

## 5.5 The whole diagnosis on one page

`asr_predictability_panel()` composites the answer — *how much*, then *by what* — into a
single figure suitable for a report or a slide. It is also what
`pdf_asr_predictability=True` embeds into a `ratios_report()` PDF.

In [ ]:
ml.asr_predictability_panel(diag, top_n=10, n_dependence=4)

### Wiring it into the report

The same diagnostics run inside `ratios_report()`, so an automated pipeline produces them
without a separate step.

In [ ]:
ml_report = rs.ratios_report(
    sample_df,
    assessment_header=ASSESSMENT,
    market_value_header=MARKET_VALUE,
    group_by='property_type',
    descriptive=True,
    model_name='DemoModel_ML',
    export_directory=results_directory,
    export_in_compacted_file='',
    return_dict=False,
    pdf_args={'pdf_vertical_equity_panel': True,
              'pdf_asr_predictability': True,
              'asr_predictability_variables': CHARACTERISTICS,
              'asr_predictability_group_by': 'neighbourhood',
              'asr_predictability_shap_sample_pct': 0.20})

sorted(p.name for p in results_directory.glob('*DemoModelML*.pdf'))

---
# 7. Sales chasing detection

Everything in sections 2 to 6 rests on one assumption, and the Standard states it plainly: a ratio
study is valid *"to the extent that sold and unsold parcels are appraised at the same level
and in the same manner."*

**Sales chasing** — reappraising a property because it sold, at or near its selling price —
breaks that assumption. And it breaks it in the most dangerous possible direction: the
study looks *better*. Ratios cluster on 1.00, the COD falls, the level lands on target. The Standard
is blunt about the consequence: uniformity inferences *"will not be accurate (valuations
appear more uniform than they are)"* and measures of level *"will not be supportable"*.

This is why the Standard treats a **very low COD as a diagnostic warning** rather than a good
result, and why it devotes an entire section to detection. The package implements
it as a family:

| Function | What it needs | What it compares |
|---|---|---|
| `sc_value_change` | prior + current value, sold flag | Year-over-year value change, sold vs unsold |
| `sc_unit_values` | value, size, sold flag | Value per square foot, sold vs unsold |
| `sc_split_sample` | sale date + a cutoff | Sales the assessor could see vs could not |
| `sc_concentration` | ratios only | Share of ratios in a narrow band against the critical-concentration threshold |
| `sc_cdf_gap` | ratios only | A flat spot in the ratio CDF |
| `sc_mass_appraisal` | an independent model value | Assessment vs an independent estimate |
| `sc_independent_values` | an independent model value | The same, pointed at the oversight agency's own values |
| `detect_sales_chasing` | whatever you have | Runs everything the data supports |

The Standard asks valuers and oversight agencies to implement **at least one** of these.

## 7.1 Run everything the data supports

`detect_sales_chasing()` decides which tests it can run from the columns you pass, and
returns one tidy frame. The important design point: it reports the tests it **could not**
run, and why. "We did not test for it" and "we tested and found nothing" are different
findings, and an oversight agency reading the report has to be able to tell them apart.

In [ ]:
SC_COLUMNS = ['technique', 'reference', 'n', 'statistic', 'threshold',
              'p_value', 'flag', 'conclusion']

screens = rs.detect_sales_chasing(df=sample_df,
                                  assessment_col=ASSESSMENT,
                                  sale_price_col=MARKET_VALUE)
screens[SC_COLUMNS]

With a sales file alone you get the two tests that need nothing else — the B.4
concentration screen and the CDF check. The other four report `not run` and name the column
that would enable them. Nothing here is flagged, which is the expected result for a model
whose sold and unsold parcels were valued the same way.

`group_by` runs the screens per stratum as well, because chasing is often a practice in one
office or one neighbourhood rather than jurisdiction-wide.

In [ ]:
by_type = rs.detect_sales_chasing(df=sample_df,
                                  assessment_col=ASSESSMENT,
                                  sale_price_col=MARKET_VALUE,
                                  group_by='property_type')

flagged = by_type[by_type['flag'] == True]        # noqa: E712 - None is a third state
print(f'{len(by_type)} screens run across {by_type["scope"].nunique()} scopes; '
      f'{len(flagged)} flagged\n')
flagged[['technique', 'scope', 'n', 'statistic', 'threshold', 'conclusion']]

> *"Even when critical proportions of ratios are
> exceeded, further investigation should be conducted before concluding that sales chasing
> has occurred."* A flag is a prompt to look, not a verdict — and on a small stratum a
> single duplicated ratio can trip the CDF screen.

## 7.2 What detection looks like when there *is* something to find

The honest way to demonstrate a detector is to give it something to detect. Here is the
same sales file with 30% of the sales reassessed to their own sale price — the textbook
chase.

In [ ]:
rng = np.random.default_rng(7)
chased = sample_df.copy()
hit = rng.choice(chased.index, size=int(len(chased) * 0.30), replace=False)
chased.loc[hit, ASSESSMENT] = chased.loc[hit, MARKET_VALUE] * rng.normal(1.0, 0.004, len(hit))

# The study now looks *better* - which is exactly the problem.
comparison = pd.DataFrame({
    'clean':  rs.ratios(sample_df, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                        indicators=['Median', 'CoD', 'PRD'], print_ci_settings=False).iloc[0],
    'chased': rs.ratios(chased, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                        indicators=['Median', 'CoD', 'PRD'], print_ci_settings=False).iloc[0],
})
comparison.loc[['Count', 'Median', 'CoD', 'PRD']]

The COD drops sharply and the median moves onto 1.00. Judged on the compliance table alone
this is the better model. It is not — it is the same model with a third of its evidence
contaminated.

Now the screens:

In [ ]:
rs.detect_sales_chasing(df=chased,
                        assessment_col=ASSESSMENT,
                        sale_price_col=MARKET_VALUE)[SC_COLUMNS]

The concentration test catches it decisively: 40% of ratios fall within ±2% of the median, against the 32%
the Standard says is the most a genuine sample with a 4% COD should produce.

`critical_concentration()` is that table, callable. Given the lowest COD credibly achievable
for a property type, it returns the share of ratios above which chasing is the likely
explanation.

In [ ]:
cods = (0.016, 0.04, 0.05, 0.06, 0.08, 0.10)
table = pd.DataFrame([rs.critical_concentration(min_achievable_cod=c) for c in cods],
                     index=[f'{c:.1%}' for c in cods])
table.index.name = 'minimum achievable COD'
table.rename(columns={'sd': 'implied SD',
                      'expected_proportion': 'expected share within ±2%',
                      'critical_proportion': 'critical share'})

## 7.3 The roll-based tests

B.1 and B.2 are the tests that speak to the mechanism rather than a symptom, and they need
the **unsold** population — which an assessor has and an oversight agency usually does not.

The sample file is sales only, so the cell below builds a synthetic roll: a sold flag, a
prior value, and a current value in which the unsold were not revalued at all while the
sold were set to their sale price.

In [ ]:
roll = sample_df.copy()
roll['sold'] = False
roll.loc[np.random.default_rng(3).choice(roll.index, 3000, replace=False), 'sold'] = True

roll['prior_value'] = roll[PRIOR_MODEL]
roll['current_value'] = roll[PRIOR_MODEL]                       # unsold: no revaluation
roll.loc[roll['sold'], 'current_value'] = roll.loc[roll['sold'], MARKET_VALUE]   # sold: chased

b1 = rs.sc_value_change(roll, sold_col='sold',
                        prior_value_col='prior_value', current_value_col='current_value')
pd.Series({k: v for k, v in b1.items() if k != 'detail'}).to_frame('B.1 value change')

Two refinements are visible in that output:

- **`statistic` switched to `mean`.** The Standard prefers the median, but notes that *"if this
  proportion is high in the unsold group, the median value change in that group may not be
  meaningful, in which case the mean should be used."* Here `unsold_pct_no_change` is 100%,
  so the median change is 0 for everyone and carries no information. The function noticed.
- **`statistically_significant` and `practically_significant` are reported separately.**
  The Standard warns that on large samples *"small differences … can be proven to be statistically
  significant, yet the actual differences may be slight"*, and sets a 5% tolerance on top of
  the p-value. Both must clear before anything is flagged.

The unit-value test needs only one year of values plus a size column, which is often the difference between
a test you can run and one you cannot.

In [ ]:
b2 = rs.sc_unit_values(roll, sold_col='sold',
                       value_col='current_value', size_col='living_area')
pd.Series({k: v for k, v in b2.items() if k != 'detail'}).to_frame('B.2 unit values')

## 7.4 The split-sample test

B.3 is the one technique an oversight agency can run on a sales file alone. It splits the
sales at the last date the assessor could have acted on them: sales *before* the cutoff
could have been chased, sales *after* it could not. Adjusted for time, the two halves should
agree.

The sample file carries `sale_month` rather than a date, so the cell builds one.

In [ ]:
dated = sample_df.assign(
    sale_date=pd.to_datetime('2022-01-01')
              + pd.to_timedelta((sample_df['sale_month'] - 1) * 30, unit='D'))

b3 = rs.sc_split_sample(dated,
                        assessment_col=ASSESSMENT,
                        sale_price_col=MARKET_VALUE,
                        sale_date_col='sale_date',
                        cutoff_date='2023-10-01')     # last date values could change

pd.Series({k: v for k, v in b3.items() if k != 'detail'}).to_frame('B.3 split sample')

The verdict is *Inconclusive*, and the wording is worth reading: the two subsamples do
differ significantly, but no statistic is **both** better in the could-have-been-chased half
**and** outside its tolerance. Chasing makes the earlier half look *better*; a difference
that is merely present, or that runs the wrong way, is not evidence of it.

## 7.5 Where this belongs in a study

Run a screen every cycle, not only when something looks wrong — the whole hazard is that
chasing makes a study look right. In practice:

1. **Before trusting a low COD.** The Standard treats a COD below the range as a
   warning sign; a concentration screen is the cheapest way to check it.
2. **Before publishing.** The Standard asks the report to describe methodology and the proper use of
   results. A line recording which screens ran and what they found belongs there.
3. **Whenever the sample was not built by you.** The Standard lists the influence of sales
   chasing as *"difficult to detect or prevent"* among the disadvantages of using
   assessor-supplied sales.

And if a screen does flag, the Standard sets out the remedy: redo the study on a sample period
that precludes chasing — typically sales occurring *after* the last date values could be
changed, time-adjusted back to the valuation date — before any reappraisal or equalization
decision is taken.

---
**Next:** [05 Vertical equity](05_vertical_equity.ipynb).